In [34]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException   
from datetime import datetime, timedelta
import traceback
import pandas as pd
import numpy as np
from time import sleep
import urllib.parse
import json
import os

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 30)
pd.set_option('future.no_silent_downcasting', True)

In [35]:
# Set up functions
def print_update(func):
    def wrapper(*args, **kwargs):
        print(f'Running {func.__name__}')
        return func(*args, **kwargs)
    return wrapper


# def print_update(func):
#     def wrapper(*args, **kwargs):
#         start_time = datetime.now()
#         print(f'Running {func.__name__}')
#         result = func(*args, **kwargs)
#         end_time = datetime.now()
#         duration = end_time - start_time
#         print(f'{func.__name__} took {duration} to run')
#         return result
#     return wrapper


def reject_cookies(driver, temp=0, retries=2):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.ID, 'onetrust-reject-all-handler')))
        driver.find_element(By.ID, 'onetrust-reject-all-handler').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{retries} attempts failed to Reject cookies')
        else:
            reject_cookies(driver, temp + 1)


def skip_tutorial(driver, temp=0, retries=2):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip')))
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{temp} attempts failed to Skip the tutorial')
        else:
            skip_tutorial(driver, temp + 1)

@print_update
def setup_driver(url):
    options = webdriver.ChromeOptions()
    options.set_capability("goog:loggingPrefs", {"performance": "INFO"})
    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)
    options.add_argument("--no-sandbox")
    options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-extensions")
    options.add_argument('--ignore-certificate-errors')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-dev-shm-usage')
    driver = webdriver.Chrome(options=options)

    driver.get(url)

    reject_cookies(driver)
    skip_tutorial(driver)

    return driver


def get_explorer_urls(product_id=None, category_id=None, object_id=None):
    # gets the corresponding urls to wallapop listing explorer
    base = 'https://es.wallapop.com/app/search?'
    urls = []

    if product_id:
        pass ## Do later
        # url = url if (product_id == None) else url + f'keywords={urllib.parse.quote(product_id)}&'
    else:
        url = base + f'object_type_ids={object_id}&category_ids={category_id}&filters_source=quick_filters&latitude=40.41956&longitude=-3.69196&'
        start, increment, loops = 0, 4, 12
        current_range = increment

        for i in range(loops):
            min = start
            max = start + current_range

            if i == 0:
                urls.append((f'{min}-{max}',(url + f'max_sale_price={max}')))
            elif i + 1 == len(range(loops)):
                urls.append((f'{min+.01}-{max}',(url + f'min_sale_price={min+.01}')))
            else:
                urls.append((f'{min+.01}-{max}',(url + f'min_sale_price={min+.01}&max_sale_price={max}')))

            start += current_range
            current_range += increment + (i-1)**2 # Ramps up range ~according to previous scrapes

    return urls


@print_update
def load_more(driver, temp=0, retries=2):
    # Clicks button that initiates infinite scroll dynamic listings
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.ID, 'btn-load-more')))
        driver.find_element(By.ID, 'btn-load-more').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{temp} attempts failed to Click the load button')
        else:
            load_more(driver, temp + 1)


def wait_listings(driver, temp=0, retries=3):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.ItemCardList__item')))
        return True
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{temp} attempts failed to Find listings')
        else:
            wait_listings(driver, temp + 1)


def convert_timestamp(ts):
    return datetime.fromtimestamp(ts / 1000).strftime('%y%m%d%H')

In [36]:
# parsing functions
# import emoji
# from nltk.corpus import stopwords
# from nltk.tokenize import word_tokenize, sent_tokenize
# from langdetect import detect
# from langdetect.lang_detect_exception import LangDetectException


# LANGUAGE_MAP = {
#     'en': 'english',
#     'es': 'spanish',
#     'fr': 'french',
#     'de': 'german',
#     'it': 'italian',
#     'pt': 'portuguese',
#     'nl': 'dutch',
#     'ru': 'russian',
#     'zh-cn': 'chinese',
#     'ja': 'japanese',
# }


# def detect_language(text):
#     if text:
#         try:
#             return detect(text)
#         except LangDetectException:
#             return 'es'
#     return 'none'


# def calculate_char_metrics(text):
#     if text:
#         lowercase_char_count = 0
#         capitalized_char_count = 0
#         digit_count = 0
#         emoji_count = 0
#         non_alphanumeric_char_count = 0
#         line_break_count = 0
#         whitespace_count = 0
        
#         for char in text:
#             if char.islower():
#                 lowercase_char_count += 1
#             if char.isupper():
#                 capitalized_char_count += 1
#             if char.isdigit():
#                 digit_count += 1

#             if char in emoji.EMOJI_DATA:
#                 emoji_count += 1
#             elif not char.isalnum() and not char.isspace():
#                 non_alphanumeric_char_count += 1

#             if char == '\n':
#                 line_break_count += 1
#             elif char.isspace():
#                 whitespace_count += 1

#         metrics = {
#         'lowercase_char_count': lowercase_char_count,
#         'capitalized_char_count': capitalized_char_count,
#         'digit_count': digit_count,
#         'emoji_count': emoji_count,
#         'non_alphanumeric_char_count': non_alphanumeric_char_count,
#         'line_break_count': line_break_count,
#         'whitespace_count': whitespace_count,
#         }
#         return metrics
#     else:
#         return {
#         'lowercase_char_count': 0,
#         'capitalized_char_count': 0,
#         'digit_count': 0,
#         'emoji_count': 0,
#         'non_alphanumeric_char_count': 0,
#         'line_break_count': 0,
#         'whitespace_count': 0,
#         }


# def calculate_metrics(text, lang_code):
#     if text:
#         language = LANGUAGE_MAP.get(lang_code, 'spanish')
#         sentences = sent_tokenize(text)
#         words = [word for word in word_tokenize(text) if word.isalpha()]
#         stop_words = set(stopwords.words(language))
        
#         allcaps_word_count = 0
#         total_word_length = 0
#         unique_words = set()
#         stop_word_count = 0
        
#         for word in words:
#             if word.isupper():
#                 allcaps_word_count += 1
#             total_word_length += len(word)
#             unique_words.add(word)
#             if word.lower() in stop_words:
#                 stop_word_count += 1

#         avg_sent_len = round((len(words) / len(sentences) if sentences else 0), 2)
#         avg_word_len = round((total_word_length / len(words) if len(words) else 0), 2)
        
#         metrics = {
#             'sentence_count': len(sentences),
#             'avg_sent_len_cents': int(avg_sent_len*100),
#             'allcaps_word_count': allcaps_word_count,
#             'word_count': len(words),
#             'avg_word_len_cents': int(avg_word_len*100),
#             'unique_word_count': len(unique_words),
#             'stop_word_count': stop_word_count,
#         }
#         return metrics
#     else:
#         return {
#             'sentence_count': 0,
#             'avg_sent_len_cents': 0,
#             'allcaps_word_count': 0,
#             'word_count': 0,
#             'avg_word_len_cents': 0,
#             'unique_word_count': 0,
#             'stop_word_count': 0,
#         }


def convert_timestamp(ts):
    return datetime.fromtimestamp(ts / 1000).strftime('%y%m%d%H')

In [37]:
# Scraping functions
def verify_filepath(filepath):
    directory = os.path.dirname(filepath)
    if not os.path.exists(directory):
        os.makedirs(directory)


def update_scraping(json_df, category_id=None, object_id=None, range_name=None):
    global scraped_df

    try:
        filepath = f'data/{category_id}/{object_id}/{category_id}_{object_id}_price:{range_name}.csv'
        object_df = pd.read_csv(filepath)
        for _, row in json_df.iterrows():
            match = object_df['id'] == row['id']
            if match.any():
                for col in object_df.columns.intersection(json_df.columns):
                    object_df.loc[match, col] = row[col]
            else:
                object_df = pd.concat([object_df, pd.DataFrame([row])], ignore_index=True)
        
        object_df.to_csv(filepath, index=False)
        scraped_df = pd.concat([scraped_df, json_df])  
        return json_df

    except FileNotFoundError:
        object_df = json_df
        verify_filepath(filepath)
        object_df.to_csv(filepath, index=False)
        scraped_df = pd.concat([scraped_df, json_df])
        return json_df


def parse_json(response_json):
    df = pd.DataFrame(response_json['data']['section']['payload']['items'])

    # Split up shipping
    shipping_normalized = pd.json_normalize(df['shipping'])
    df = df.reset_index(drop=True)
    shipping_normalized = shipping_normalized.reset_index(drop=True)
    df = pd.concat([df, shipping_normalized], axis=1)

    # # Get language and word metrics
    # df['lang_code'] = df['description'].apply(detect_language)
    # metrics_df = df.apply(lambda row: calculate_metrics(row['description'], row['lang_code']), axis=1).apply(pd.Series)
    # metrics_df = metrics_df.astype({
    #     'allcaps_word_count': 'int',
    #     'word_count': 'int',
    #     'unique_word_count': 'int',
    #     'stop_word_count': 'int',
    #     'sentence_count': 'int',
    # })
    # df = pd.concat([df, metrics_df], axis=1)

    # # Get char metrics
    # metrics_df = df['description'].apply(calculate_char_metrics).apply(pd.Series)
    # df = pd.concat([df, metrics_df], axis=1)
    
    # Get time metrics
    df['date_last_modified'] = df['modified_at'].apply(lambda ts: int(datetime.fromtimestamp(ts / 1000).strftime('%y%m%d%H')))
    df['date_first_created'] = df['created_at'].apply(lambda ts: int(datetime.fromtimestamp(ts / 1000).strftime('%y%m%d%H')))
    df['date_last_scraped'] = int(datetime.now().strftime('%y%m%d%H'))

    # Clean up othe metrics
    df['country_code'] = df['location'].apply(lambda x: x['country_code'])
    df['num_images'] = df['images'].apply(lambda x:len(x))
    df['refurbished'] = df['is_refurbished'].apply(lambda x: list(x.values())[0])
    df['reserved'] = df['reserved'].apply(lambda x: list(x.values())[0])
    df['currency'] = df['price'].apply(lambda x: list(x.values())[1])
    df['price_cents'] = df['price'].apply(lambda x: int(list(x.values())[0]*100))

    replace_dict = {'none': 0,
                    True: 1,
                    False: 0}
    df['refurbished'] = df['refurbished'].replace(replace_dict)
    df['reserved'] = df['reserved'].replace(replace_dict)
    df['user_allows_shipping'] = df['user_allows_shipping'].replace(replace_dict)
    df['item_is_shippable'] = df['item_is_shippable'].replace(replace_dict)

    # df = df.drop(['images', 'shipping', 'price', 'discount', 'favorited', 'cost_configuration_id', 'taxonomy', 'location', 'favoriteable', 'created_at', 'modified_at', 'is_refurbished', 'is_favoriteable', 'category_id', 'bump', 'description'], axis=1, errors='ignore') # For when we activate description parsing
    
    df = df.drop(['images', 'shipping', 'price', 'discount', 'favorited', 'cost_configuration_id', 'taxonomy', 'location', 'favoriteable', 'created_at', 'modified_at', 'is_refurbished', 'is_favoriteable', 'category_id', 'bump'], axis=1, errors='ignore')

    return df


def get_response_body(driver, request_id, temp=0, retries=20):
    try:
        response_body = driver.execute_cdp_cmd("Network.getResponseBody", {"requestId": request_id})
        return response_body
    except Exception:
        print(temp)
        if temp >= retries:
            print(f'{temp} attemps failed to get json response body')
        else:
            sleep(temp/1000)
            return get_response_body(driver, request_id, temp + 1)


def get_json(driver, timed_out, timer=None, category_id=None, object_id=None, range_name=None):
    # global json_df
    scrapings_df = pd.DataFrame()
    
    logs = driver.get_log("performance")
    while logs:
        for i, log in enumerate(logs):
            log_json = json.loads(log["message"])["message"]
            if log_json["method"] == 'Network.requestWillBeSent':
                try:
                    response_url = log_json["params"]["documentURL"]
                    if response_url.startswith("https://api.wallapop.com/api/v3/search?"):
                        request_id = log_json["params"]["initiator"]["requestId"]
                        try:
                            response_body = get_response_body(driver, request_id)
                            response_json = json.loads(response_body['body'])
                            json_df = parse_json(response_json)
                            json_df = update_scraping(json_df, category_id=category_id, object_id=object_id, range_name=range_name)
                            scrapings_df = pd.concat([scrapings_df, json_df]).drop_duplicates()

                        except Exception as e:
                            print(f'GET_JSON EXCEPTION: {e}')
                            traceback.print_exc()

                except KeyError:
                    print('KEYERROR IN GET JSON')
                    print(f'\n\n{log_json}')
                    
        logs = driver.get_log("performance")

    clear_explorer(driver)

    # Print stats
    timespent = datetime.now() - timer
    efficiency = round(len(scrapings_df)/timespent.total_seconds(), 3)
    print(f'    scrapings took: {timespent}')
    print(f'    scrapings count: {len(scrapings_df)}')
    print(f'    scrapings efficiency = {efficiency}/sec')

    if timed_out:
        return scrapings_df, True
    else:
        return scroll_explorer(driver, category_id=category_id, object_id=object_id, range_name=range_name)


def scroll_explorer(driver, category_id=None, object_id=None, range_name=None):
    global scraped_df
    global max_scrolls
    previous_height = driver.execute_script("return document.body.scrollHeight")
    sleep_time = 0.1
    count = 0
    timer = datetime.now()
    complete = False

    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        if count == max_scrolls:
            print('Scroll: Count reached')
            # sleep(sleep_time/2)
            scraped_df, complete = get_json(driver, timed_out=False, timer=timer, category_id=category_id, object_id=object_id, range_name=range_name)

        try:
            WebDriverWait(driver, sleep_time).until(lambda driver: driver.execute_script("return document.body.scrollHeight") != previous_height)
            count += 1
            sleep_time /= 2
            previous_height = driver.execute_script("return document.body.scrollHeight")
        except TimeoutException:
            sleep_time += sleep_time
            if sleep_time > 1.5:
                print('Scroll: Timed out')
                try:
                    card = driver.find_elements(By.CSS_SELECTOR, 'a.ItemCardList__item')[-1]
                    WebDriverWait(driver, 5).until(EC.element_to_be_clickable(card))
                except Exception as e:
                    print(f'scroll {e}')
                scraped_df, complete = get_json(driver, timed_out=True, timer=timer, category_id=category_id, object_id=object_id, range_name=range_name)

        if complete:
            return scraped_df, True


def clear_explorer(driver):
    driver.execute_script("window.scrollTo(0, 0);")
    try:
        # Find elements and delete all siblings
        element = driver.find_element(By.CSS_SELECTOR, 'a.ItemCardList__item')
        parent = element.find_element(By.XPATH, '..')
        siblings = parent.find_elements(By.XPATH, '*')
        for sibling in siblings:
            driver.execute_script("arguments[0].remove();", sibling)
    except NoSuchElementException:
        print('    No listings found to clear to clear')
   

def check_categories(category_ids=[24200, 12579, 13100], product_id=None):
    global max_scrolls
    global scraped_df
    all_scraped = 0

    for category_id in category_ids:
        scraped_count = 0
        category_start = datetime.now()

        try:
            categories_path = 'data/categories.csv'
            categories = pd.read_csv(categories_path)
            object_type_ids = categories[categories['category_id'] == category_id]
            object_type_ids = object_type_ids[object_type_ids['subgroup_name'] != 'FULL_GROUP']
        except FileNotFoundError:
            verify_filepath(categories_path)
            return f'{categories_path} is missing'
        category_name = categories.loc[categories['category_id'] == category_id, 'category_name'].values[0]
        print(f'\n=== Checking {category_name}')

        for object_id in object_type_ids['object_type_id']:
            object_name = categories.loc[categories['object_type_id'] == object_id, 'subgroup_name'].values[0]
            object_start = datetime.now()
            urls = get_explorer_urls(product_id=product_id, category_id=category_id, object_id=object_id)
            
            try:
                urls_path = f'data/{category_id}/{object_id}/{object_id}_urls.csv'
                urls_df = pd.read_csv(urls_path)
            except FileNotFoundError:
                urls_df = pd.DataFrame(urls)
                urls_df['last_checked'] = 0
                urls_df = urls_df.rename(columns={0:'range_name', 1:'url'})
                verify_filepath(urls_path)
                urls_df.to_csv(urls_path)

            urls_df = urls_df[urls_df['last_checked'] != int(object_start.strftime('%y%m%d%H'))]
            
            scraped_df = pd.DataFrame()
            for _, row in urls_df.iterrows():
                range_name = row['range_name']
                url = row['url']
                print(f'\nSCRAPING {object_name.upper()} {object_id} LOOP {range_name} \n')
                complete = False
                driver = setup_driver(url + '&order_by=price_high_to_low')
                if wait_listings(driver):
                    try:
                        load_more(driver)
                        scraped_df, complete = scroll_explorer(driver, category_id=category_id, object_id=object_id, range_name=range_name)
                        driver.quit()
                        print(f'loop:{range_name} - high_to_low - {object_name} {object_id} scraped successfully')
                    except Exception as e:
                        traceback.print_exc()
                        driver.quit()
                        print(f'loop:{range_name} - high_to_low - {object_name} {object_id} ERROR: {e}')

                    if not complete:
                        driver = setup_driver(url + '&order_by=price_low_to_high')
                        if wait_listings(driver):
                            try:
                                load_more(driver)
                                scraped_df, _ = scroll_explorer(driver, category_id=category_id, object_id=object_id, range_name=range_name)
                                driver.quit()
                                print(f'loop:{range_name} - low_to_high - {object_name} {object_id} scraped successfully')
                            except Exception as e:
                                traceback.print_exc()
                                driver.quit()
                                print(f'loop:{range_name} - low_to_high - {object_name} {object_id} ERROR: {e}')
                        else:
                            driver.quit()
                else:
                    driver.quit()

                # Update categories.csv
                urls_df.loc[urls_df['range_name'] == range_name, 'last_checked'] = int(datetime.now().strftime('%y%m%d%H'))
                urls_df.to_csv(urls_path, index=False)
                print(f'\n{object_name.upper()} {object_id} LOOP {range_name} COMPLETE\n')

            timespent = datetime.now() - object_start
            scraped_count += len(scraped_df)
            efficiency = round(len(scraped_df)/timespent.total_seconds(), 3)

            print(f'\n{object_name.upper()} {object_id} SCRAPE COMPLETE')
            print(f'    {object_name} took: {timespent}')
            print(f'    {object_name} scraped: {len(scraped_df)}')
            print(f'    {object_name} efficency: {efficiency}/sec')

        # Update fullgroup in category.csv
        categories.loc[(categories['category_id'] == category_id) & (categories['subgroup_name'] == 'FULL_GROUP'), 'last_checked'] = int(datetime.now().strftime('%y%m%d%H'))
        categories.to_csv('data/categories.csv', index=False)

        timespent = datetime.now() - category_start
        all_scraped += scraped_count
        print(f'\n{category_name} CATEGORY checked')
        print(f'    {category_name} took: {timespent}')
        print(f'    {category_name} scraped: {scraped_count}')
        print(f'    {category_name} efficency: {round(scraped_count/timespent.total_seconds(), 3)}/sec')

    if all_scraped == 0:
        return True
    print('\n=== All categories checked ===\n')

In [38]:
categories = [24200]
max_scrolls = 2
scraped_df = pd.DataFrame()
# json_df = pd.DataFrame()
 
while True:
    try:
        if check_categories(categories):
            now = datetime.now()
            next_hour = (now + timedelta(hours=1)).replace(minute=0, second=0, microsecond=0)
            time_to_sleep = (next_hour - now).total_seconds()
            sleep(time_to_sleep)
    except Exception:
        print('error outside')
        traceback.print_exc()
        break


=== Checking Tecnología y electrónica

SCRAPING TV 10414 LOOP 4.01-13 

Running setup_driver
Running load_more
Scroll: Count reached
    scrapings took: 0:00:01.901453
    scrapings count: 158
    scrapings efficiency = 83.094/sec
Scroll: Count reached
0
1
2
3
4
5
6
7
    scrapings took: 0:00:01.696325
    scrapings count: 120
    scrapings efficiency = 70.741/sec
Scroll: Count reached
0
1
2
3
4
    scrapings took: 0:00:01.709500
    scrapings count: 119
    scrapings efficiency = 69.611/sec
Scroll: Count reached
0
1
2
3
4
5
6
7
8
    scrapings took: 0:00:01.715424
    scrapings count: 120
    scrapings efficiency = 69.954/sec
Scroll: Count reached
0
1
2
3
4
5
    scrapings took: 0:00:01.722489
    scrapings count: 120
    scrapings efficiency = 69.667/sec
Scroll: Count reached
0
1
2
3
4
5
6
7
    scrapings took: 0:00:01.749679
    scrapings count: 120
    scrapings efficiency = 68.584/sec
Scroll: Count reached
0
1
2
3
4
5
6
    scrapings took: 0:00:01.657924
    scrapings count: 117
